# Build the modelling dataset: void features + thermal class label

Combines the two tables produced by the earlier notebooks into one clean table for model training and testing.

| input | made by | content |
|---|---|---|
| `rth_start_end.csv` | `jp_data_exploration.ipynb` | Rth, cycles-to-fail and the class `label` for each LED (target) |
| `../data/processed/spatial_void_features.csv` | `jp_void_spatial_features.ipynb` | spatial void features from the 0-cycle X-ray image of each LED (inputs) |

Both tables have one row per LED and share the key `unit_id` = `<LED type>_<paste>_P<panel>_L<LED no.>`.

**Rows are removed when they cannot be used for training or testing:**
1. no thermal data or no class label (no TTA measurement, or the LED was removed from the test before it failed),
2. no X-ray image, so no void features (missing file or a "Missing" placeholder picture).

**Output:** `../data/processed/model_dataset.csv`.

**Label:** 1 = Survivor (20 % increase in Rth never reached by 1500 cycles), 2 = Mid-life failure (500 ≤ `tsc_fail` ≤ 1500), 3 = Early failure (`tsc_fail` < 500). A failure first seen at 1500 cycles is class 2 (Part II §V.C); see `jp_data_exploration.ipynb` section 3.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RTH_CSV = Path("rth_start_end.csv")
FEAT_CSV = Path("../data/processed/spatial_void_features.csv")
OUT_CSV = Path("../data/processed/model_dataset.csv")

rth = pd.read_csv(RTH_CSV)
feat = pd.read_csv(FEAT_CSV)
print(f"Rth / label table : {rth.shape[0]} LEDs, {rth.shape[1]} columns")
print(f"void feature table: {feat.shape[0]} LEDs, {feat.shape[1]} columns")
assert rth.unit_id.is_unique and feat.unit_id.is_unique, "unit_id must identify exactly one LED in each table"

Rth / label table : 1800 LEDs, 18 columns
void feature table: 1799 LEDs, 33 columns


## 1. Merge

Outer join on `unit_id` first, so that every LED of either table is visible and the reason for dropping it can be counted. The feature table calls the LED number `led_num`; it is the same as `led` in the Rth table, so only one is kept after checking that the identification columns agree.

In [2]:
m = rth.merge(feat.rename(columns={"led_num": "led"}), on="unit_id", how="outer",
              suffixes=("", "_feat"), indicator="source")

# the identification columns must agree wherever both tables have the LED
both = m.source == "both"
for c in ["led_type", "paste", "panel", "led"]:
    assert (m.loc[both, c] == m.loc[both, c + "_feat"]).all(), f"{c} differs between the two tables"
    m[c] = m[c].fillna(m[c + "_feat"])
m = m.drop(columns=[c + "_feat" for c in ["led_type", "paste", "panel", "led"]])

print(m.source.value_counts().rename({"both": "in both tables", "left_only": "only in the Rth table (no X-ray file)",
                                      "right_only": "only in the feature table"}).to_string())

source
in both tables                           1799
only in the Rth table (no X-ray file)       1
only in the feature table                   0


## 2. Remove rows that cannot be used

Each LED gets exactly one reason, checked in this order: no thermal data → no label → no X-ray features → usable.

In [3]:
def why_dropped(r):
    if pd.isna(r.rth_start):
        return "no Rth data (no TTA measurement)"
    if pd.isna(r.label):
        return "no label (removed from the test before failing)"
    if r.source != "both":
        return "no X-ray image file"
    if r.image_missing == 1:
        return "X-ray image is a 'Missing' placeholder"
    return "usable"

m["status"] = m.apply(why_dropped, axis=1)
account = pd.crosstab(m.led_type, m.status)
account.loc["TOTAL"] = account.sum()
print(account.to_string())

data = m[m.status == "usable"].drop(columns=["status", "source", "image_missing"]).reset_index(drop=True)
data["label"] = data["label"].astype(int)
print(f"\nkept {len(data)} of {len(m)} LEDs")

status    X-ray image is a 'Missing' placeholder  no Rth data (no TTA measurement)  no label (removed from the test before failing)  usable
led_type                                                                                                                                   
FC-GB1                                         0                               200                                                0       0
FC-GB2                                         0                                41                                                5     154
FC-GB3                                         2                                 0                                                5     193
FC-SP1                                         0                                 0                                               20     180
FC-SP2                                         2                                 0                                                9     189
FC-SP3              

## 3. Fill the feature values that are undefined by construction

The remaining empty cells are not missing measurements. They are features that have no value for that LED:

- **`thermal_*` on 2-pad LEDs** (FC-SP1, FC-SP2, FC-GB3): these packages have no thermal pad. The column `has_thermal_pad` (0/1) already says so.
- **`*_mean_circularity` and `*_edge_proximity_min` when a pad has no void**: there is no void whose shape or distance could be measured.

They are filled with the value that means "no void", so that every model (including SVM, which cannot take empty cells) can use the table:

| feature | filled with | reason |
|---|---|---|
| `*_edge_proximity_min` | 1 | scale is 0 = void at the pad edge, 1 = as far from the edge as possible |
| every other void feature | 0 | no void area, no cluster, no shape |

In [4]:
ID_COLS = ["unit_id", "led_type", "paste", "panel", "led"]
TARGET = "label"
# measured during / after thermal cycling: describe the outcome, so they must NOT be used as model inputs
OUTCOME_COLS = ["label_name", "label_status", "tsc_fail", "tsc_fail_sustained", "last_cycle", "rth_500", "rth_end",
                "rth_increase", "rth_increase_pct", "ri_500_pct", "ri_max_pct"]
INFO_COLS = ["rth_start"]            # measured before cycling; not a void feature, kept for reference
FEATURE_COLS = [c for c in data.columns if c not in ID_COLS + [TARGET] + OUTCOME_COLS + INFO_COLS]

before = data[FEATURE_COLS].isna().sum()
edge_cols = [c for c in FEATURE_COLS if c.endswith("edge_proximity_min")]
data[edge_cols] = data[edge_cols].fillna(1.0)
data[FEATURE_COLS] = data[FEATURE_COLS].fillna(0.0)

print("cells filled per feature:")
print(before[before > 0].to_string())
print(f"\nempty cells left in the features: {int(data[FEATURE_COLS].isna().sum().sum())}")

cells filled per feature:
thermal_void_area_frac                562
thermal_centroid_x                    562
thermal_centroid_y                    562
thermal_q_NW                          562
thermal_q_NE                          562
thermal_q_SW                          562
thermal_q_SE                          562
thermal_cluster_count                 562
thermal_mean_circularity              609
thermal_largest_void_frac             562
thermal_edge_proximity_min            609
electrical_mean_mean_circularity      108
electrical_mean_edge_proximity_min    108

empty cells left in the features: 0


## 4. Final checks

In [5]:
assert data.unit_id.is_unique, "duplicate LEDs"
assert data[FEATURE_COLS + [TARGET]].notna().all().all(), "empty cells left"
assert np.isfinite(data[FEATURE_COLS].to_numpy(dtype=float)).all(), "non-finite feature values"
assert set(data[TARGET].unique()) <= {1, 2, 3}
assert ((data[TARGET] == 1) == data.tsc_fail.isna()).all(), "Survivor must mean the 20 % increase was never reached"

constant = [c for c in FEATURE_COLS if data[c].nunique() <= 1]
print(f"{len(data)} LEDs, {len(FEATURE_COLS)} features, target = '{TARGET}'")
print("features that are the same for every LED:", constant if constant else "none")

LABEL_NAME = {1: "1 Survivor", 2: "2 Mid-life failure", 3: "3 Early failure"}
counts = pd.crosstab(data.led_type, data[TARGET].map(LABEL_NAME))
counts["total"] = counts.sum(axis=1)
counts.loc["TOTAL"] = counts.sum()
print("\nclass counts per LED type:")
print(counts.to_string())
print("\nclass share [%]:", (100 * data[TARGET].value_counts(normalize=True).sort_index()).round(1).to_dict())

1470 LEDs, 27 features, target = 'label'
features that are the same for every LED: none

class counts per LED type:
label     1 Survivor  2 Mid-life failure  3 Early failure  total
led_type                                                        
FC-GB2             0                 139               15    154
FC-GB3             0                 156               37    193
FC-SP1           177                   0                3    180
FC-SP2            17                 145               27    189
FC-SP3            23                 149               19    191
FC-SP4             0                 178               14    192
VTF1             180                   0                0    180
VTF2              18                 166                7    191
TOTAL            415                 933              122   1470

class share [%]: {1: 28.2, 2: 63.5, 3: 8.3}


## 5. Save

Column order in `model_dataset.csv`:

1. **identification** — `unit_id`, `led_type`, `paste`, `panel`, `led`
2. **features** (`FEATURE_COLS`) — the spatial void features, plus `has_thermal_pad`, `pad_px`, `void_count`, `void_px`, `void_ratio`
3. **target** — `label` (1 = Survivor, 2 = Mid-life failure incl. failure at 1500 cycles, 3 = Early failure)
4. **reference only, not model inputs** — `rth_start`, then the outcome columns (`label_name`, `label_status`, `tsc_fail`, Rth after cycling, relative increases). The outcome columns are measured during or after thermal cycling; using them as inputs would leak the answer into the model.

`led_type` and `paste` are categorical inputs for the baseline model (one-hot encode them when training).

In [6]:
data = data[ID_COLS + FEATURE_COLS + [TARGET] + INFO_COLS + OUTCOME_COLS]
data.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}: {data.shape[0]} rows x {data.shape[1]} columns")
print("\nFEATURE_COLS =", FEATURE_COLS)
data.head()

wrote ..\data\processed\model_dataset.csv: 1470 rows x 45 columns

FEATURE_COLS = ['has_thermal_pad', 'thermal_void_area_frac', 'thermal_centroid_x', 'thermal_centroid_y', 'thermal_q_NW', 'thermal_q_NE', 'thermal_q_SW', 'thermal_q_SE', 'thermal_cluster_count', 'thermal_mean_circularity', 'thermal_largest_void_frac', 'thermal_edge_proximity_min', 'electrical_mean_void_area_frac', 'electrical_mean_centroid_x', 'electrical_mean_centroid_y', 'electrical_mean_q_NW', 'electrical_mean_q_NE', 'electrical_mean_q_SW', 'electrical_mean_q_SE', 'electrical_mean_cluster_count', 'electrical_mean_mean_circularity', 'electrical_mean_largest_void_frac', 'electrical_mean_edge_proximity_min', 'pad_px', 'void_count', 'void_px', 'void_ratio']


,unit_id,led_type,paste,panel,led,has_thermal_pad,thermal_void_area_frac,thermal_centroid_x,thermal_centroid_y,thermal_q_NW,...,label_status,tsc_fail,tsc_fail_sustained,last_cycle,rth_500,rth_end,rth_increase,rth_increase_pct,ri_500_pct,ri_max_pct
0,FC-GB2_SAC107-BiIn_P1_L02,FC-GB2,SAC107-BiIn,1,2,1.0,0.020928,0.615373,0.072432,0.000000,...,ok,500.0,500.0,1500.0,12.084356,29.839828,20.264891,211.645181,26.208211,211.645181
1,FC-GB2_SAC107-BiIn_P1_L03,FC-GB2,SAC107-BiIn,1,3,1.0,0.010285,-0.529327,0.457574,0.174419,...,ok,750.0,750.0,1500.0,10.744296,16.605415,7.029735,73.412388,12.204009,73.412388
2,FC-GB2_SAC107-BiIn_P1_L04,FC-GB2,SAC107-BiIn,1,4,1.0,0.045802,0.497724,0.012773,0.000000,...,ok,750.0,750.0,1500.0,9.556831,14.151400,4.538046,47.205646,-0.587963,47.205646
3,FC-GB2_SAC107-BiIn_P1_L05,FC-GB2,SAC107-BiIn,1,5,1.0,0.004186,-0.166625,0.228571,0.000000,...,ok,500.0,500.0,1500.0,11.691541,16.517620,6.980075,73.185239,22.584383,73.185239
4,FC-GB2_SAC107-BiIn_P1_L06,FC-GB2,SAC107-BiIn,1,6,1.0,0.002511,0.026549,0.486486,0.000000,...,ok,500.0,500.0,1500.0,12.173268,17.237209,7.815018,82.942673,29.197835,82.942673
